In [25]:
import numpy as np
from scipy.special import jv
import matplotlib.pyplot as plt
from scipy.io import loadmat
import h5py

In [26]:
# load the images
with h5py.File('assignmentImageDenoising_microscopy.mat', 'r') as f:
    noisy_image = np.array(f['microscopyImageNoisyScale350sigma0point06'])
    noiseless_image = np.array(f['microscopyImageOrig'])

# Normalize images
max_value = 255
noiseless_image = noiseless_image / max_value
noisy_image = noisy_image / max_value

print(noiseless_image[:, 0, 0])

<KeysViewHDF5 ['microscopyImageNoisyScale350sigma0point06', 'microscopyImageOrig']>
[0.63137255 0.5372549  0.58431373]


In [35]:
  # squared L2 norm
def prior1(data, col, depth, pixel_value, gamma):
    nd1, nd2, nd3 = data.shape
    total = 0

    for direction in [(1,0),(-1,0),(0,1),(0,-1)]:
        new_col, new_depth = (col + direction[0]) % nd2, (depth + direction[1]) % nd3
        diff = pixel_value - data[:, new_col, new_depth]
        total += np.linalg.norm(diff, ord=2)

    return total ** 2


# L2 norm
def prior2(data, col, depth, pixel_value, gamma):
    nd1, nd2, nd3 = data.shape
    total = 0

    for direction in [(1,0),(-1,0),(0,1),(0,-1)]:
        new_col, new_depth = (col + direction[0]) % nd2, (depth + direction[1]) % nd3
        diff = pixel_value - data[:, new_col, new_depth]
        total += np.linalg.norm(diff, ord=2)

    return total

# Huber regularized L1 norm
def prior3(data, col, depth, pixel_value, gamma):
    nd1, nd2, nd3 = data.shape
    total = 0

    for direction in [(1,0), (0,1), (-1,0), (0,-1)]:
        new_col, new_depth = (col + direction[0]) % nd2, (depth + direction[1]) % nd3
        norm = np.linalg.norm(pixel_value - data[:, new_col, new_depth], ord=1)

        if norm <= gamma:
            total += norm
        else:
            total += gamma * norm - 0.5 * gamma ** 2

    return total

In [28]:
# compute the prior values for all pixels, index through col, depth
def compute_prior_matrix(data, prior_func, gamma):
    nd1, nd2, nd3 = data.shape
    prior_values = np.zeros((nd2, nd3))

    for col in range(nd2):
        for depth in range(nd3):
            prior_values[col, depth] = prior_func(data, col, depth, data[:, col, depth], gamma)
    return prior_values

In [29]:
# to compute the gradient
def compute_gradient(prior_func, alpha, data_noisy, estimate, gamma):
    return (1 - alpha) * (estimate - data_noisy) + alpha * compute_prior_matrix(estimate, prior_func, gamma) # sus

In [30]:
# relative root mean squared error
def rrmse(matrix1, matrix2):
    return np.sqrt(np.sum((matrix1 - matrix2)**2)) / np.sqrt(np.sum(matrix1**2))

In [31]:
def posterior(noisy_data, estimate, prior_func, alpha, gamma):
    likelihood_term = -0.5 * np.sum((estimate - noisy_data) ** 2)

    prior_term = -alpha * np.sum(compute_prior_matrix(estimate, prior_func, gamma))

    return likelihood_term + prior_term

In [ ]:
# gradient descent
def gradient_descent(noisy_data, prior_func, alpha, gamma, max_iter=30):
    estimate = noisy_data.copy()

    learning_rate = 0.3
    pos_vals = []

    for iteration in range(max_iter):
        step_size = learning_rate / (1 + iteration/10)

        gradient = compute_gradient(prior_func, alpha, noisy_data, estimate, gamma)
        new_estimate = estimate - step_size * gradient

        new_estimate = np.clip(new_estimate, 0, 1)

        change = np.mean(np.abs(new_estimate - estimate))
        estimate = new_estimate

        log_posterior = posterior(noisy_data, estimate, prior_func, alpha, gamma)
        pos_vals.append(log_posterior)


        current_rrmse = rrmse(noiseless_image, estimate)
        print(f"Iteration {iteration}, RRMSE: {current_rrmse:.6f}, Change: {change:.6f}")

        # early termination
        if change < 1e-6:
            print("Converged")
            break

    return estimate, pos_vals

In [33]:
# to plot
def plot_results(noiseless, noisy, denoised, alpha, gamma, rrmse_value):
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))

    axes[0, 0].imshow(noiseless, cmap='gray', vmin=0, vmax=1)
    axes[0, 0].set_title('Noiseless Image')

    axes[0, 1].imshow(noisy, cmap='gray', vmin=0, vmax=1)
    axes[0, 1].set_title('Noisy Image')

    axes[1, 0].imshow(denoised, cmap='gray', vmin=0, vmax=1)
    axes[1, 0].set_title(f'Denoised Image (Prior)\nRRMSE: {rrmse_value:.4f}\nα={alpha}, γ={gamma}')

    diff = np.abs(noiseless - denoised)
    axes[1, 1].imshow(diff, cmap='hot', vmin=0, vmax=0.5)
    axes[1, 1].set_title('Difference Map')

    plt.tight_layout()
    plt.show()

In [36]:
# set up parameters
alpha = 0.18
gamma = 0.25

denoised = gradient_descent(noisy_image, prior3, 1.2*alpha, gamma)
final_rrmse = rrmse(noiseless_image, denoised)

print("RRMSE : ", final_rrmse)

# plot_results(noiseless_image, noisy_image, denoised, alpha, gamma, final_rrmse)

Iteration 0, RRMSE: 0.189843, Change: 0.009905
Iteration 5, RRMSE: 0.199180, Change: 0.004719
Iteration 10, RRMSE: 0.207749, Change: 0.002788
Iteration 15, RRMSE: 0.214287, Change: 0.001858
Iteration 20, RRMSE: 0.219294, Change: 0.001338
Iteration 25, RRMSE: 0.223221, Change: 0.001014
Iteration 30, RRMSE: 0.226377, Change: 0.000802
Iteration 35, RRMSE: 0.228970, Change: 0.000651
Iteration 40, RRMSE: 0.231137, Change: 0.000543
Iteration 45, RRMSE: 0.232977, Change: 0.000461
Iteration 50, RRMSE: 0.234561, Change: 0.000398
Iteration 55, RRMSE: 0.235942, Change: 0.000348
Iteration 60, RRMSE: 0.237154, Change: 0.000307
Iteration 65, RRMSE: 0.238229, Change: 0.000274
Iteration 70, RRMSE: 0.239189, Change: 0.000247
Iteration 75, RRMSE: 0.240053, Change: 0.000224
Iteration 80, RRMSE: 0.240834, Change: 0.000204
Iteration 85, RRMSE: 0.241546, Change: 0.000188
Iteration 90, RRMSE: 0.242197, Change: 0.000173
Iteration 95, RRMSE: 0.242794, Change: 0.000161
Iteration 100, RRMSE: 0.243344, Change: 0.

In [44]:
def plot_all_results(noiseless, noisy, denoised1, denoised2, denoised3, params):
    """
    Plot the results of image denoising.
    Option 1: Using transpose for RGB display
    """
    # Option 1: Transpose images from (3, H, W) to (H, W, 3)
    noiseless = np.transpose(noiseless, (1, 2, 0))
    noisy = np.transpose(noisy, (1, 2, 0))
    denoised1 = np.transpose(denoised1, (1, 2, 0))
    denoised2 = np.transpose(denoised2, (1, 2, 0))
    denoised3 = np.transpose(denoised3, (1, 2, 0))

    # OR Option 2: Take just one channel for grayscale
    # noiseless = noiseless[0]
    # noisy = noisy[0]
    # denoised1 = denoised1[0]
    # denoised2 = denoised2[0]
    # denoised3 = denoised3[0]

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    plt.subplots_adjust(wspace=0.3, hspace=0.3)

    # Find global min and max for consistent colormap scaling
    all_images = [noiseless, noisy, denoised1, denoised2, denoised3]
    vmin = min(image.min() for image in all_images)
    vmax = max(image.max() for image in all_images)

    # Plot noiseless image
    im0 = axes[0, 0].imshow(noiseless, cmap='jet', vmin=vmin, vmax=vmax)
    axes[0, 0].set_title('Noiseless Image')
    axes[0, 0].axis('off')

    # Plot noisy image
    axes[0, 1].imshow(noisy, cmap='jet', vmin=vmin, vmax=vmax)
    axes[0, 1].set_title('Noisy Image')
    axes[0, 1].axis('off')

    # Plot denoised image with prior1 (Quadratic)
    axes[0, 2].imshow(denoised1, cmap='jet', vmin=vmin, vmax=vmax)
    axes[0, 2].set_title(f'Prior 1 (Quadratic)\nRRMSE: {params["rrmse1"]:.4f}\nα={params["alpha1"]:.3f}, γ={params["gamma1"]:.3f}')
    axes[0, 2].axis('off')

    # Plot denoised image with prior2 (Huber)
    axes[1, 0].imshow(denoised2, cmap='jet', vmin=vmin, vmax=vmax)
    axes[1, 0].set_title(f'Prior 2 (Huber)\nRRMSE: {params["rrmse2"]:.4f}\nα={params["alpha2"]:.3f}, γ={params["gamma2"]:.3f}')
    axes[1, 0].axis('off')

    # Plot denoised image with prior3 (Discontinuity-adaptive)
    axes[1, 1].imshow(denoised3, cmap='jet', vmin=vmin, vmax=vmax)
    axes[1, 1].set_title(f'Prior 3 (Discontinuity-adaptive)\nRRMSE: {params["rrmse3"]:.4f}\nα={params["alpha3"]:.3f}, γ={params["gamma3"]:.3f}')
    axes[1, 1].axis('off')

    # Remove the empty subplot
    axes[1, 2].remove()

    # Add colorbar
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    fig.colorbar(im0, cax=cbar_ax, label='Pixel Value')

    # Save and show the plot
    plt.savefig('report_c.png', bbox_inches='tight', dpi=300)
    plt.show()

In [ ]:
params = {
        'alpha1': 0.11, 'gamma1': 0.01, 'rrmse1': 0.281,
        'alpha2': 0.18, 'gamma2': 0.25, 'rrmse2': 0.265,
        'alpha3': 0.512, 'gamma3': 0.032, 'rrmse3': 0.242 # find optimal
    }

# # Get denoised images using optimal parameters
denoised1, pos1 = gradient_descent(noisy_image, prior1, params['alpha1'], params['gamma1'])
denoised2, pos2 = gradient_descent(noisy_image, prior2, params['alpha2'], params['gamma2'])
denoised3, pos3 = gradient_descent(noisy_image, prior3, params['alpha3'], params['gamma3'])

# Create the comparison plot
plot_all_results(noiseless_image, noisy_image, denoised1, denoised2, denoised3, params)

Iteration 0, RRMSE: 0.189348, Change: 0.006005
Iteration 1, RRMSE: 0.190164, Change: 0.004995
Iteration 2, RRMSE: 0.191128, Change: 0.004237
Iteration 3, RRMSE: 0.192159, Change: 0.003650
Iteration 4, RRMSE: 0.193212, Change: 0.003187
Iteration 5, RRMSE: 0.194260, Change: 0.002813
Iteration 6, RRMSE: 0.195289, Change: 0.002507
Iteration 7, RRMSE: 0.196290, Change: 0.002253
Iteration 8, RRMSE: 0.197262, Change: 0.002040
Iteration 9, RRMSE: 0.198202, Change: 0.001859
Iteration 10, RRMSE: 0.199109, Change: 0.001704
Iteration 11, RRMSE: 0.199986, Change: 0.001570
Iteration 12, RRMSE: 0.200835, Change: 0.001453
Iteration 13, RRMSE: 0.201658, Change: 0.001351
Iteration 14, RRMSE: 0.202457, Change: 0.001262
Iteration 15, RRMSE: 0.203233, Change: 0.001182
Iteration 16, RRMSE: 0.203988, Change: 0.001111
Iteration 17, RRMSE: 0.204726, Change: 0.001048
Iteration 18, RRMSE: 0.205449, Change: 0.000992
Iteration 19, RRMSE: 0.206154, Change: 0.000940
Iteration 20, RRMSE: 0.206840, Change: 0.000893
It

In [46]:
# Plot objective functions vs iterations
def plot_posterior(pos_vals1, pos_vals2, pos_vals3):
    # plt.subplot(3,1,1)
    plt.figure(figsize=(10, 7))

    plt.plot(pos_vals1, marker='o', linestyle='-')
    plt.xlabel("Iteration")
    plt.ylabel("-ve Log Posterior Value")
    plt.title("Prior 1")

    # plt.subplot(3,1, 2)

    plt.figure(figsize=(10, 10))
    plt.plot(pos_vals2, marker='o', linestyle='-')

    plt.xlabel("Iteration")
    plt.ylabel("Log Posterior Value")
    plt.title("Prior 2")

    # plt.subplot(3,1,3)
    plt.figure(figsize=(10, 10))
    plt.plot(pos_vals3, marker='o', linestyle='-')

    plt.xlabel("Iteration")
    plt.ylabel("Log Posterior Value")
    plt.title("Prior 3")

    plt.tight_layout()
    # plt.grid()
    plt.show()

In [ ]:
plot_posterior(pos1, pos2, pos3)